<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/07_agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Agentic RAG

## Notebook 07 — Real Multi-Agent Investigation & Verification

This notebook validates MetricGuard's production Agentic RAG workflow using:

- real Qdrant dense retrieval
- mandatory Cross-Encoder reranking
- deterministic metric governance metadata
- Gemini-powered Metric Investigation
- Gemini-powered Verification & Reporting
- LangGraph conditional routing
- bounded self-correction
- evidence-grounded final conclusions

### Agent Workflow

Question  
→ Evidence Retrieval Agent  
→ Metric Investigation Agent  
→ Verification & Reporting Agent  
→ Approved / Revise / Insufficient Evidence

If revision is required:

Verification Agent  
→ Metric Investigation Agent  
→ Verification Agent

Maximum revisions: **1**

In [63]:
from pathlib import Path
import shutil
import subprocess

GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/metricguard-ai.git"
)

REPO_DIR = Path(
    "/content/metricguard-ai"
)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

print("Repository:", REPO_DIR)

Repository: /content/metricguard-ai


## Fetching latest production code

In [64]:
import subprocess

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "pull",
        "origin",
        "main",
    ],
    check=True,
)

print("✅ Latest MetricGuard production code pulled.")

✅ Latest MetricGuard production code pulled.


In [65]:
%pip install -q -e "/content/metricguard-ai"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricguard-ai (pyproject.toml) ... done


In [66]:
from qdrant_client import (
    QdrantClient,
    models,
)

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

import langgraph

print("✅ qdrant-client ready.")
print("✅ sentence-transformers ready.")
print("✅ LangGraph ready.")

✅ qdrant-client ready.
✅ sentence-transformers ready.
✅ LangGraph ready.


In [67]:
import sys

SOURCE_DIR = (
    REPO_DIR
    / "src"
)

if str(SOURCE_DIR) not in sys.path:
    sys.path.append(
        str(SOURCE_DIR)
    )

print(SOURCE_DIR)

/content/metricguard-ai/src


In [68]:
from metricguard.agents import (
    EvidenceRetrievalAgent,
    MetricInvestigator,
    MetricInvestigationAgent,
    VerificationReporter,
    VerificationReportingAgent,
    MetricGuardAgentSystem,
    build_metricguard_agent_graph,
    load_agent_config,
)

print("✅ Production agent modules imported.")

✅ Production agent modules imported.


In [69]:
agent_config = load_agent_config(
    REPO_DIR
)

print("✅ Agent config loaded.")
print("Max revisions:", agent_config.max_revisions)

✅ Agent config loaded.
Max revisions: 1


## Load API Key

In [70]:
from google.colab import userdata
from google import genai

# Load Gemini API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get(
    "GEMINI_API_KEY"
)

assert GEMINI_API_KEY, (
    "GEMINI_API_KEY was not found "
    "in Colab Secrets."
)

# Model used by MetricGuard
GEMINI_MODEL = "gemini-3.7-flash"

# Create Gemini client
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini client created.")
print("Model:", GEMINI_MODEL)

✅ Gemini client created.
Model: gemini-3.7-flash


In [71]:
import time

# ---------------------------------------------------------
# GEMINI RATE LIMIT SETTINGS
# ---------------------------------------------------------

# AI Studio currently shows 5 requests per minute.
GEMINI_RPM_LIMIT = 5

# 60 / 5 = 12 seconds.
# Add 2 seconds as a safety buffer.
MIN_GEMINI_INTERVAL = 14

_last_gemini_call = 0.0


# ---------------------------------------------------------
# DEVELOPMENT CACHE
# ---------------------------------------------------------

# Stores already-generated answers during this Colab session.
# If the exact same question is asked again,
# Gemini will NOT be called again.
_gemini_answer_cache = {}


# ---------------------------------------------------------
# RATE LIMIT HELPER
# ---------------------------------------------------------

def wait_for_gemini_slot():
    global _last_gemini_call

    elapsed = (
        time.monotonic()
        - _last_gemini_call
    )

    wait_time = max(
        0,
        MIN_GEMINI_INTERVAL - elapsed,
    )

    if wait_time > 0:
        print(
            f"⏳ Gemini rate-limit safety wait: "
            f"{wait_time:.1f} seconds"
        )

        time.sleep(
            wait_time
        )

    _last_gemini_call = (
        time.monotonic()
    )


print("✅ Gemini rate limiter ready.")
print("✅ Development cache ready.")

✅ Gemini rate limiter ready.
✅ Development cache ready.


In [72]:
wait_for_gemini_slot()

interaction = (
    gemini_client
    .interactions
    .create(
        model=GEMINI_MODEL,
        input=(
            "Reply with exactly: "
            "GEMINI_OK"
        ),
    )
)

print(
    interaction.output_text
)

GEMINI_OK


In [73]:
from metricguard.llm import (
    build_structured_llm,
)

agent_llm = build_structured_llm(
    repo_root=REPO_DIR,
    client=gemini_client,
    minimum_request_interval_seconds=14.0,
)

print("✅ Shared production Gemini adapter ready.")

✅ Shared production Gemini adapter ready.


## Regenerating Chunks

In [74]:
from datetime import date

from metricguard.lineage.enrichment_pipeline import (
    run_full_knowledge_enrichment,
)

run_full_knowledge_enrichment(
    REPO_DIR,
    as_of_date=date(
        2026,
        8,
        18,
    ),
)

METRICGUARD FULL KNOWLEDGE ENRICHMENT REPORT
Parsed documents      : 55
Final chunks          : 167
Metric-aware chunks   : 97
Lineage-aware chunks  : 90
Lineage graph nodes   : 33
Lineage graph edges   : 30
Freshness as-of       : 2026-08-18
Ground truth          : excluded
Embedding readiness   : YES


In [75]:
import json

CHUNKS_PATH = (
    REPO_DIR
    / "data"
    / "processed"
    / "fully_enriched_chunks.jsonl"
)

chunks = []

with CHUNKS_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    for line in file:
        chunks.append(
            json.loads(line)
        )

print(
    "Chunks:",
    len(chunks)
)

Chunks: 167


In [76]:
assert not any(
    "ground_truth"
    in chunk[
        "metadata"
    ].get(
        "source_path",
        "",
    )
    for chunk in chunks
)

print(
    "✅ Ground truth excluded."
)

✅ Ground truth excluded.


In [77]:
from metricguard.retrieval import (
    CrossEncoderReranker,
    DenseRetriever,
    RetrievalPipeline,
    format_final_evidence,
    load_embedding_model,
    load_reranker_model,
    load_retrieval_config,
)

retrieval_config = (
    load_retrieval_config(
        REPO_DIR
    )
)

retrieval_config

RetrievalConfig(candidate_top_k=20, final_top_k=5, reranker_model='cross-encoder/ms-marco-MiniLM-L6-v2', embedding_model='sentence-transformers/all-mpnet-base-v2', collection_name='metricguard_dense_v1', normalize_embeddings=True)

In [78]:
embedding_model = (
    load_embedding_model(
        retrieval_config
        .embedding_model
    )
)

VECTOR_SIZE = (
    embedding_model
    .get_embedding_dimension()
)

print(
    "Vector size:",
    VECTOR_SIZE
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector size: 768


In [79]:
def build_embedding_text(
    chunk: dict,
) -> str:

    metadata = chunk[
        "metadata"
    ]

    parts = [
        (
            "Source type: "
            f"{metadata.get('source_type')}"
        ),
        (
            "Asset type: "
            f"{metadata.get('asset_type')}"
        ),
        (
            "File: "
            f"{metadata.get('file_name')}"
        ),
    ]

    for label, key in [
        ("Metric", "metric_name"),
        (
            "Observed version",
            "observed_version",
        ),
        (
            "Authoritative version",
            "authoritative_version",
        ),
        (
            "Version relation",
            "version_relation",
        ),
        (
            "Freshness",
            "freshness_status",
        ),
    ]:

        value = metadata.get(key)

        if value:
            parts.append(
                f"{label}: {value}"
            )

    return (
        "\n".join(parts)
        + "\n\n"
        + chunk["content"]
    )

In [80]:
embedding_texts = [
    build_embedding_text(
        chunk
    )
    for chunk in chunks
]

embeddings = (
    embedding_model.encode(
        embedding_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
)

print(
    embeddings.shape
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(167, 768)


In [81]:
from qdrant_client import (
    QdrantClient,
    models,
)

qdrant_client = (
    QdrantClient(
        ":memory:"
    )
)

COLLECTION_NAME = (
    retrieval_config
    .collection_name
)

qdrant_client.create_collection(
    collection_name=
        COLLECTION_NAME,
    vectors_config=
        models.VectorParams(
            size=VECTOR_SIZE,
            distance=
                models.Distance.COSINE,
        ),
)

print(
    "✅ Qdrant ready."
)

✅ Qdrant ready.


In [82]:
import uuid

points = []

for chunk, vector in zip(
    chunks,
    embeddings,
):

    point_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            chunk["chunk_id"],
        )
    )

    payload = {
        "chunk_id":
            chunk["chunk_id"],
        "content":
            chunk["content"],
        **chunk["metadata"],
    }

    points.append(
        models.PointStruct(
            id=point_id,
            vector=
                vector.tolist(),
            payload=payload,
        )
    )

In [83]:
qdrant_client.upsert(
    collection_name=
        COLLECTION_NAME,
    points=points,
    wait=True,
)

print(
    "Qdrant points:",
    qdrant_client
    .get_collection(
        COLLECTION_NAME
    )
    .points_count,
)

Qdrant points: 167


## Building Production Retriever

In [84]:
production_dense = (
    DenseRetriever(
        client=qdrant_client,
        embedding_model=
            embedding_model,
        collection_name=
            COLLECTION_NAME,
        normalize_embeddings=True,
    )
)

In [85]:
reranker_model = (
    load_reranker_model(
        retrieval_config
        .reranker_model
    )
)

production_reranker = (
    CrossEncoderReranker(
        model=reranker_model,
        batch_size=16,
    )
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [86]:
production_retrieval = (
    RetrievalPipeline(
        dense_retriever=
            production_dense,
        reranker=
            production_reranker,
        candidate_top_k=
            retrieval_config
            .candidate_top_k,
        final_top_k=
            retrieval_config
            .final_top_k,
    )
)

print(
    "✅ Production retrieval ready."
)

✅ Production retrieval ready.


## Agentic RAG Integration

In [87]:
required_objects = {
    "REPO_DIR": "REPO_DIR" in globals(),
    "gemini_client": "gemini_client" in globals(),
    "production_retrieval": "production_retrieval" in globals(),
}

for name, exists in required_objects.items():
    print(
        "✅" if exists else "❌",
        name
    )

missing = [
    name
    for name, exists
    in required_objects.items()
    if not exists
]

assert not missing, (
    "Missing prerequisite objects: "
    + ", ".join(missing)
)

print(
    "\n🔥 Phase 8.3 prerequisites are ready."
)

✅ REPO_DIR
✅ gemini_client
✅ production_retrieval

🔥 Phase 8.3 prerequisites are ready.


In [88]:
import langgraph

print(
    "✅ Editable MetricGuard package refreshed."
)
print(
    "✅ LangGraph import successful."
)

✅ Editable MetricGuard package refreshed.
✅ LangGraph import successful.


In [89]:
agent_config = load_agent_config(
    REPO_DIR
)

print(
    "✅ Agent configuration loaded."
)

print(
    "Max revisions:",
    agent_config.max_revisions
)

✅ Agent configuration loaded.
Max revisions: 1


In [90]:
agent_llm = build_structured_llm(
    repo_root=REPO_DIR,
    client=gemini_client,
    minimum_request_interval_seconds=14.0,
)

print(
    "✅ Shared Gemini agent LLM ready."
)

✅ Shared Gemini agent LLM ready.


### Building Agent 1

In [91]:
evidence_agent = EvidenceRetrievalAgent(
    retrieval_pipeline=
        production_retrieval
)

print(
    "✅ Agent 1: Evidence Retrieval ready."
)

✅ Agent 1: Evidence Retrieval ready.


### Building Agent 2

In [92]:
investigator = MetricInvestigator(
    llm=agent_llm
)

investigation_agent = (
    MetricInvestigationAgent(
        investigator=investigator
    )
)

print(
    "✅ Agent 2: Metric Investigation ready."
)

✅ Agent 2: Metric Investigation ready.


### Building Agent 3

In [93]:
verifier = VerificationReporter(
    llm=agent_llm
)

verification_agent = (
    VerificationReportingAgent(
        verifier=verifier
    )
)

print(
    "✅ Agent 3: Verification & Reporting ready."
)

✅ Agent 3: Verification & Reporting ready.


### Compiling Langgraph

In [94]:
agent_graph = (
    build_metricguard_agent_graph(
        evidence_agent=
            evidence_agent,

        investigation_agent=
            investigation_agent,

        verification_agent=
            verification_agent,
    )
)

print(
    "✅ MetricGuard LangGraph compiled."
)

✅ MetricGuard LangGraph compiled.


In [95]:
agent_system = (
    MetricGuardAgentSystem(
        graph=agent_graph,
        max_revisions=
            agent_config.max_revisions,
    )
)

print(
    "🔥 MetricGuard Agent System ready."
)

🔥 MetricGuard Agent System ready.


### Validation

In [96]:
checks = {
    "REPO_DIR":
        REPO_DIR is not None,

    "Gemini client":
        gemini_client is not None,

    "Production retrieval":
        production_retrieval is not None,

    "Shared agent LLM":
        agent_llm is not None,

    "Agent 1":
        evidence_agent is not None,

    "Agent 2":
        investigation_agent is not None,

    "Agent 3":
        verification_agent is not None,

    "LangGraph":
        agent_graph is not None,

    "Agent system":
        agent_system is not None,
}


for name, ok in checks.items():

    print(
        "✅" if ok else "❌",
        name
    )


assert all(
    checks.values()
)

print(
    "\n🔥 FULL PHASE 8.3 ASSEMBLY PASSED."
)

✅ REPO_DIR
✅ Gemini client
✅ Production retrieval
✅ Shared agent LLM
✅ Agent 1
✅ Agent 2
✅ Agent 3
✅ LangGraph
✅ Agent system

🔥 FULL PHASE 8.3 ASSEMBLY PASSED.


In [97]:
agentic_result_cache = {}


def run_agent_question(question: str):
    normalized_question = " ".join(
        question.strip().split()
    )

    if normalized_question in agentic_result_cache:
        print(
            "✅ Using cached agentic result. "
            "No new Gemini calls."
        )

        return agentic_result_cache[
            normalized_question
        ]

    print(
        "🔥 Running real MetricGuard "
        "agentic investigation..."
    )

    result = agent_system.investigate(
        normalized_question
    )

    agentic_result_cache[
        normalized_question
    ] = result

    return result


print(
    "✅ Notebook agentic cache ready."
)

✅ Notebook agentic cache ready.


In [98]:
import json


def show_agent_result(result):
    print("=" * 80)
    print("FINAL DECISION")
    print("=" * 80)

    print(
        result.get(
            "verification_decision"
        )
    )

    print("\n" + "=" * 80)
    print("REVISION COUNT")
    print("=" * 80)

    print(
        result.get(
            "revision_count",
            0
        )
    )

    print("\n" + "=" * 80)
    print("LANGGRAPH TRACE")
    print("=" * 80)

    for step in result.get(
        "trace",
        []
    ):
        print("->", step)

    print("\n" + "=" * 80)
    print("AGENT 2 INVESTIGATION")
    print("=" * 80)

    print(
        json.dumps(
            result.get(
                "investigation",
                {}
            ),
            indent=2,
            default=str
        )
    )

    print("\n" + "=" * 80)
    print("AGENT 3 FINAL REPORT")
    print("=" * 80)

    print(
        json.dumps(
            result.get(
                "final_report",
                {}
            ),
            indent=2,
            default=str
        )
    )

In [99]:
def validate_agent_result(
    result,
    label: str
):
    decision = result.get(
        "verification_decision"
    )

    report = result.get(
        "final_report",
        {}
    )

    evidence = result.get(
        "evidence",
        []
    )

    revision_count = int(
        result.get(
            "revision_count",
            0
        )
    )

    assert result.get(
        "retrieval_complete"
    ) is True

    assert decision in {
        "approved",
        "insufficient_evidence",
    }

    assert (
        revision_count
        <= agent_config.max_revisions
    )

    assert (
        report.get("decision")
        == decision
    )

    confidence = float(
        report.get(
            "confidence",
            0.0
        )
    )

    assert 0.0 <= confidence <= 1.0

    allowed_ids = {
        f"E{i}"
        for i in range(
            1,
            len(evidence) + 1
        )
    }

    used_ids = set(
        report.get(
            "evidence_ids",
            []
        )
    )

    assert used_ids.issubset(
        allowed_ids
    )

    if decision == "approved":
        assert used_ids

    for item in evidence:
        source_path = str(
            item.get(
                "source_path",
                ""
            )
        )

        assert (
            "ground_truth"
            not in source_path
        )

    print(
        f"✅ {label}: structural validation passed."
    )

## Realtime Agent Check

### Revenue

In [100]:
revenue_question = (
    "Why does the Executive KPI Dashboard report "
    "different Net Revenue from the Finance Revenue "
    "Dashboard after April 1, 2026?"
)

revenue_agent_result = (
    run_agent_question(
        revenue_question
    )
)

🔥 Running real MetricGuard agentic investigation...


#### Inspecting Results

In [101]:
show_agent_result(
    revenue_agent_result
)

FINAL DECISION
approved

REVISION COUNT
0

LANGGRAPH TRACE
-> evidence_retrieval_agent
-> metric_investigation_agent
-> verification_reporting_agent

AGENT 2 INVESTIGATION
{
  "diagnosis": "metric_migration",
  "metric_name": "net_revenue",
  "hypothesis": "The discrepancy in Net Revenue after April 1, 2026, is caused by an ongoing metric migration where the Finance Revenue Dashboard has migrated to Net Revenue v3 (deducting posted chargebacks) while the Executive KPI Dashboard remains on v2.",
  "findings": [
    "Beginning April 1, 2026, Northstar Commerce established Net Revenue v3 as the official definition, requiring the deduction of posted chargebacks across all reporting assets (E5).",
    "Finance Analytics completed the migration of the Finance Revenue Dashboard and Finance Daily mart to Net Revenue v3 (E1, E3).",
    "The Executive KPI Dashboard and pipeline have not yet been migrated from v2, resulting in higher reported Net Revenue on dates with chargeback activity because 

In [104]:
validate_agent_result(
    revenue_agent_result,
    "Net Revenue"
)

revenue_report = (
    revenue_agent_result[
        "final_report"
    ]
)

revenue_diagnosis = (
    revenue_report[
        "diagnosis"
    ]
)

print(
    "Net Revenue diagnosis:",
    revenue_diagnosis
)

assert (
    revenue_agent_result[
        "verification_decision"
    ]
    == "approved"
)

assert (
    revenue_diagnosis
    in {
        "metric_migration",
        "version_mismatch",
        "stale_definition",
    }
), (
    "Unexpected Net Revenue diagnosis: "
    f"{revenue_diagnosis}"
)

print(
    "✅ Net Revenue semantic validation passed."
)

✅ Net Revenue: structural validation passed.
Net Revenue diagnosis: metric_migration
✅ Net Revenue semantic validation passed.


### Total Orders

In [105]:
orders_question = (
    "The Operations Dashboard and Finance Dashboard "
    "report different Total Orders. Is this a data "
    "pipeline failure, or are the dashboards "
    "intentionally using different metric definitions?"
)

orders_agent_result = (
    run_agent_question(
        orders_question
    )
)

🔥 Running real MetricGuard agentic investigation...


#### Inspecting Results

In [106]:
show_agent_result(
    orders_agent_result
)

FINAL DECISION
approved

REVISION COUNT
0

LANGGRAPH TRACE
-> evidence_retrieval_agent
-> metric_investigation_agent
-> verification_reporting_agent

AGENT 2 INVESTIGATION
{
  "diagnosis": "intentional_semantic_difference",
  "metric_name": "total_orders",
  "hypothesis": "The difference in Total Orders between the Operations Dashboard and Finance Dashboard is driven by intentional semantic differences rather than a data pipeline failure: Operations counts all placed orders (excluding cancelled) to track workload, whereas the enterprise/Finance definition counts successfully paid orders.",
  "findings": [
    "E1 explicitly notes that Operations and Finance intentionally use distinct order-counting concepts (placed orders excluding cancelled orders vs. successfully paid orders) and confirms this should not be classified as a pipeline failure.",
    "E2 and E3 document closed incident INC-003 regarding the discrepancy between Operations Dashboard and Finance Revenue Dashboard, concludin

In [107]:
validate_agent_result(
    orders_agent_result,
    "Total Orders"
)

orders_report = (
    orders_agent_result[
        "final_report"
    ]
)

assert (
    orders_agent_result[
        "verification_decision"
    ]
    == "approved"
)

assert (
    orders_report[
        "diagnosis"
    ]
    == "intentional_semantic_difference"
)

print(
    "✅ Total Orders semantic validation passed."
)

✅ Total Orders: structural validation passed.
✅ Total Orders semantic validation passed.


### Active Customers

In [108]:
active_customer_question = (
    "Why might the Growth Dashboard's Active Customers "
    "number disagree with the current authoritative "
    "Active Customers definition?"
)

active_customer_result = (
    run_agent_question(
        active_customer_question
    )
)

🔥 Running real MetricGuard agentic investigation...


#### Inspecting Results

In [109]:
show_agent_result(
    active_customer_result
)

FINAL DECISION
approved

REVISION COUNT
0

LANGGRAPH TRACE
-> evidence_retrieval_agent
-> metric_investigation_agent
-> verification_reporting_agent

AGENT 2 INVESTIGATION
{
  "diagnosis": "version_mismatch",
  "metric_name": "active_customers",
  "hypothesis": "The Growth & Marketing Dashboard displays an Active Customers count that disagrees with the authoritative definition because it is using metric version v1 (an engagement-based definition counting customers with recent digital/website/mobile activity), whereas the authoritative enterprise Active Customers definition is v2 (requiring at least one successfully paid order in the past 30 days).",
  "findings": [
    "E1: Customer Analytics review notes that the enterprise Active Customer definition changed on March 1, 2026 to require at least one successfully paid order in the previous 30 days, while Growth reporting continues counting identified customers with recent digital activity.",
    "E2 & E3: Incident INC-002 records that t

In [110]:
validate_agent_result(
    active_customer_result,
    "Active Customers"
)

active_customer_report = (
    active_customer_result[
        "final_report"
    ]
)

assert (
    active_customer_result[
        "verification_decision"
    ]
    == "approved"
)

assert (
    active_customer_report[
        "diagnosis"
    ]
    in {
        "stale_definition",
        "version_mismatch",
    }
)

print(
    "✅ Active Customers semantic validation passed."
)

✅ Active Customers: structural validation passed.
✅ Active Customers semantic validation passed.


### Unsupported Questions

In [111]:
unsupported_question = (
    "How much electricity did Northstar Commerce's "
    "warehouse air conditioning consume last Tuesday?"
)

unsupported_result = (
    run_agent_question(
        unsupported_question
    )
)

🔥 Running real MetricGuard agentic investigation...


#### Inspecting Results

In [112]:
show_agent_result(
    unsupported_result
)

FINAL DECISION
approved

REVISION COUNT
0

LANGGRAPH TRACE
-> evidence_retrieval_agent
-> metric_investigation_agent
-> verification_reporting_agent

AGENT 2 INVESTIGATION
{
  "diagnosis": "insufficient_evidence",
  "metric_name": null,
  "hypothesis": "The retrieved evidence and tool observations only contain information regarding Northstar Commerce's operational tables (E1) and net revenue metric definitions and SQL models (E2, E3, E4, E5). There is no data or evidence available concerning warehouse electricity or air conditioning consumption.",
  "findings": [
    "E1 lists raw dbt sources for customer, order, payment, refund, and web session transactions for Northstar Commerce.",
    "E2 and E3 document business rule context for versions 1 and 3 of the net_revenue metric.",
    "E4 and E5 provide deprecated SQL definitions for net_revenue v2 and v1.",
    "No evidence, tables, or metrics related to warehouse facilities, HVAC, or electricity consumption exist in the available assets

In [113]:
validate_agent_result(
    unsupported_result,
    "Unsupported Question"
)

unsupported_decision = (
    unsupported_result[
        "verification_decision"
    ]
)

unsupported_relevance_gap = (
    unsupported_decision
    != "insufficient_evidence"
)

if unsupported_relevance_gap:
    print(
        "⚠️ Unsupported question was not rejected."
    )

    print(
        "⚠️ Phase 8.4 needs a retrieval "
        "relevance gate."
    )

else:
    print(
        "✅ Unsupported question correctly "
        "returned insufficient evidence."
    )

✅ Unsupported Question: structural validation passed.
⚠️ Unsupported question was not rejected.
⚠️ Phase 8.4 needs a retrieval relevance gate.


### Final Checks

In [114]:
phase83_results = {
    "Net Revenue":
        revenue_agent_result,

    "Total Orders":
        orders_agent_result,

    "Active Customers":
        active_customer_result,

    "Unsupported":
        unsupported_result,
}


print("=" * 100)
print("PHASE 8.3 — REAL AGENTIC RAG RESULTS")
print("=" * 100)


for name, result in (
    phase83_results.items()
):
    report = result.get(
        "final_report",
        {}
    )

    print()
    print(name)

    print(
        "Decision:",
        result.get(
            "verification_decision"
        )
    )

    print(
        "Diagnosis:",
        report.get(
            "diagnosis"
        )
    )

    print(
        "Confidence:",
        report.get(
            "confidence"
        )
    )

    print(
        "Revisions:",
        result.get(
            "revision_count"
        )
    )

    print(
        "Trace:",
        " -> ".join(
            result.get(
                "trace",
                []
            )
        )
    )

PHASE 8.3 — REAL AGENTIC RAG RESULTS

Net Revenue
Decision: approved
Diagnosis: metric_migration
Confidence: 0.95
Revisions: 0
Trace: evidence_retrieval_agent -> metric_investigation_agent -> verification_reporting_agent

Total Orders
Decision: approved
Diagnosis: intentional_semantic_difference
Confidence: 0.95
Revisions: 0
Trace: evidence_retrieval_agent -> metric_investigation_agent -> verification_reporting_agent

Active Customers
Decision: approved
Diagnosis: version_mismatch
Confidence: 0.95
Revisions: 0
Trace: evidence_retrieval_agent -> metric_investigation_agent -> verification_reporting_agent

Unsupported
Decision: approved
Diagnosis: insufficient_evidence
Confidence: 1.0
Revisions: 0
Trace: evidence_retrieval_agent -> metric_investigation_agent -> verification_reporting_agent


---

## Phase 8.3 Summary — Real Agentic RAG Integration

MetricGuard's production Agentic RAG workflow has now been integrated and
validated against the real retrieval and LLM stack.

### Runtime Architecture

Question  
→ Qdrant dense retrieval  
→ top-20 candidates  
→ mandatory Cross-Encoder reranking  
→ top-5 evidence  
→ Evidence Retrieval Agent  
→ Metric Investigation Agent  
→ Verification & Reporting Agent  
→ conditional approval / revision / insufficient-evidence routing

### Production Components Validated

Agent 1 uses the real MetricGuard retrieval pipeline.

Agent 2 uses Gemini structured reasoning over retrieved evidence and
deterministic governance observations.

Agent 3 independently verifies Agent 2 against the original evidence.

LangGraph controls conditional routing and allows a bounded revision loop.

### Revision Safety

Maximum revisions:

`1`

This permits one self-correction pass while preventing uncontrolled LLM loops.

### Validation Cases

The integration includes:

1. Net Revenue version/staleness disagreement
2. Total Orders intentional semantic difference
3. Active Customers stale/current-definition disagreement
4. Unsupported out-of-domain question

### Safety Boundaries

Runtime agent evidence remains isolated from evaluation ground truth.

Agents may reference only supplied evidence IDs.

Deterministic MetricGuard code remains responsible for factual governance
metadata such as versions, freshness and lineage.

### Next

Phase 8.4 will harden the agentic system with:

- retrieval relevance gating
- final application-result assembly
- deterministic source resolution
- confidence/fallback integration
- agentic response caching
- production safeguards before evaluation